In [1]:
import pandas as pd
import numpy as np

In [2]:
music_df = pd.read_csv('01_data/processed/train.csv')
census_df = pd.read_csv('01_data/processed/acs2015_census_tract_data.csv')

In [3]:
# customer_id 삭제
if 'customer_id' in music_df.columns:
    music_df = music_df.drop('customer_id', axis=1)

In [4]:
# Census 데이터 전처리
# 결측치 처리: State별 Income의 중앙값(median)으로 채우기
census_df['Income'] = census_df.groupby('State')['Income'].transform(lambda x: x.fillna(x.median()))

# 소득을 총인구로 나누어 인구당 소득 지표 생성
census_df['Income_per_Pop'] = census_df['Income'] / census_df['TotalPop'].replace(0, np.nan)

# State별로 평균값 도출
state_stats = census_df.groupby('State').agg({
    'TotalPop': 'sum',
    'Income_per_Pop': 'mean'
}).reset_index()

# 컬럼명 변경
state_stats.columns = ['State', 'State_TotalPop', 'State_AvgIncome']

In [5]:
# 데이터 병합
# location와 State 병합
final_df = pd.merge(music_df, state_stats, left_on='location', right_on='State', how='left')

In [6]:
# 중복된 State 삭제
final_df = final_df.drop('State', axis=1)

In [7]:
# 수치 데이터 보정
final_df['average_session_length'] = final_df['average_session_length'] / 60
final_df['weekly_unique_songs'] = np.where(
    final_df['weekly_unique_songs'] > final_df['weekly_songs_played'],
    final_df['weekly_songs_played'],
    final_df['weekly_unique_songs']
)
final_df['signup_date'] = final_df['signup_date'].abs()

In [8]:
# 문자열 처리
final_df = final_df.drop('location', axis=1, errors='ignore')

cols_to_encode = ['subscription_type', 'payment_plan', 'payment_method', 'customer_service_inquiries']
final_df = pd.get_dummies(final_df, columns=cols_to_encode, drop_first=True, dtype=int)

In [9]:
# 결과 확인
print(f"최종 데이터 형태: {final_df.shape}")
display(final_df.head())

# 결측치 확인
print("병합 후 결측치 개수:\n", final_df.isnull().sum())

최종 데이터 형태: (125000, 25)


,age,num_subscription_pauses,signup_date,weekly_hours,average_session_length,song_skip_rate,weekly_songs_played,weekly_unique_songs,num_favorite_artists,num_platform_friends,...,State_AvgIncome,subscription_type_Free,subscription_type_Premium,subscription_type_Student,payment_plan_Yearly,payment_method_Credit Card,payment_method_Debit Card,payment_method_Paypal,customer_service_inquiries_Low,customer_service_inquiries_Medium
0,32,2,1606,22.391362,1.756575,0.176873,169,109,18,32,...,19.375099,1,0,0,1,0,0,1,0,1
1,64,3,2897,29.294210,0.875019,0.981811,55,55,44,33,...,26.891714,1,0,0,0,0,0,1,1,0
2,51,2,348,15.400312,0.411728,0.048411,244,117,20,129,...,17.498757,0,1,0,1,1,0,0,0,0
3,63,4,2894,22.842084,1.393258,0.035691,442,252,47,120,...,24.200499,0,0,0,1,0,0,0,0,1
4,54,3,92,23.151163,0.876304,0.039738,243,230,41,66,...,17.498757,0,0,0,0,0,0,1,0,0


병합 후 결측치 개수:
 age                                     0
num_subscription_pauses                 0
signup_date                             0
weekly_hours                            0
average_session_length                  0
song_skip_rate                          0
weekly_songs_played                     0
weekly_unique_songs                     0
num_favorite_artists                    0
num_platform_friends                    0
num_playlists_created                   0
num_shared_playlists                    0
notifications_clicked                   0
churned                                 0
State_TotalPop                       6601
State_AvgIncome                      6601
subscription_type_Free                  0
subscription_type_Premium               0
subscription_type_Student               0
payment_plan_Yearly                     0
payment_method_Credit Card              0
payment_method_Debit Card               0
payment_method_Paypal                   0
customer_service_inq